### Learning Instruction Finetuning

In [1]:
%load_ext autoreload
# %load_ext tensorboard
%autoreload 2

In [ ]:
from pathlib import Path
import os
import socket
import sys
from dotenv import load_dotenv
from pathlib import Path
import sys
import wandb


In [3]:
try:
    project_root = next(p for p in Path().cwd().resolve().parents if (p /"env.env").exists())
except StopIteration:
    # Fallback to current working directory if not found
    project_root = Path.cwd()

sys.path.insert(0, str(project_root))

print(f"Project Root: {project_root}")
env_path = (project_root / "env.env").resolve()
status = load_dotenv(dotenv_path=env_path, override=True)

print("env file:", env_path)
print("load_dotenv returned:", status)
data_dir = project_root / "datasets"

Project Root: /coc/scratch/vchopra/learning_env/Deep-Learning
env file: /coc/scratch/vchopra/learning_env/Deep-Learning/env.env
load_dotenv returned: True


In [4]:
# Check if running on the Pace server
hostname = socket.gethostname()
IS_PACE_SERVER = "inara" in hostname.lower()  # Replace 'pace' with a unique keyword from your server's hostname
IS_PACE_SERVER

True

In [5]:
import torch
from datasets import Dataset as hf_data
from torch.utils.data import Dataset
import tiktoken
import numpy, tqdm
import matplotlib.pyplot as plt
%matplotlib inline

In [6]:
## Accelerator for Pytorch
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
device

'cuda'

In [7]:

api_key = os.environ.get("WANDB_API_KEY")
assert api_key, "WANDB_API_KEY was not loaded from env.env"

wandb.finish()
wandb.login(key=api_key, relogin=True)

api = wandb.Api()
print("Logged-in W&B username:", api.viewer.username)
print("Default W&B entity:", api.default_entity)
print("WANDB_API_KEY loaded from environment:", bool(os.getenv("WANDB_API_KEY")))


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /nethome/dgarg39/.netrc
wandb: Currently logged in as: vedaangchopra (vedaangchopra_gatech) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin


Logged-in W&B username: vedaangchopra
Default W&B entity: vedaangchopra_gatech
WANDB_API_KEY loaded from environment: True


In [8]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [9]:
from model_training.framework.data.post_train.instruction_tuning.raw_data.simple_example.instruct_data import download_and_load_file
from model_training.framework.data.post_train.instruction_tuning.prompt_style.styles import SFTDatasetPrepare, AlpacaFormatOld
from model_training.framework.data.post_train.dataloader import InstructionDataset, custom_collate_fn, create_instruction_dataloader

## Fix this import

# raise Exception("Not Running on Pace")
    # from src.model import GPT

In [10]:
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    print(i, props.name, f"{props.total_memory / 1024**3:.1f} GiB")

GPU count: 8
0 NVIDIA A40 44.4 GiB
1 NVIDIA A40 44.4 GiB
2 NVIDIA A40 44.4 GiB
3 NVIDIA A40 44.4 GiB
4 NVIDIA A40 44.4 GiB
5 NVIDIA A40 44.4 GiB
6 NVIDIA A40 44.4 GiB
7 NVIDIA A40 44.4 GiB


### Step-1:-  Preparing the dataset

We are using Alpaca Format, every models have their own format, so the format of the tuning can differ. <br>
THis format is called ALPACA format{instruction, input output}... <br>
 There are different formats that exist and the data format doesn't matter, we need to take prompt style templates for instruction tuning. We need to apply those prompt templates for instruction tuning. 

In [11]:
file_name = "instruction-data.json"
url = (
    "https://raw.githubusercontent.com/rasbt/LLMs-from-scratch"
    "/main/ch07/01_main-chapter-code/instruction-data.json"
)
data = download_and_load_file(data_dir=(data_dir / "instruction_tuning"), file_name=file_name, url = url)
print("Number of entries:", len(data))
data[50] , data[999]

Number of entries: 1100


({'instruction': 'Identify the correct spelling of the following word.',
  'input': 'Ocassion',
  'output': "The correct spelling is 'Occasion.'"},
 {'instruction': "What is an antonym of 'complicated'?",
  'input': '',
  'output': "An antonym of 'complicated' is 'simple'."})

In [12]:
# alpaca = AlpacaFormatOld(data)

In [13]:
# ## Example of format
# print(alpaca.format_input(instruction=data[50]['instruction'], in_text=data[50]['input']))
# print("=============================================")
# print(alpaca.detailed_response(out_text = data[50]['output']))

In [14]:
# sample = (alpaca.complete_response(instruction=data[999]['instruction'], in_text=data[999]['input'], out_text=data[999]['output']))
# print(sample)

In [15]:
dataset  = hf_data.from_list(data)
sft_data= SFTDatasetPrepare(model_name=MODEL_NAME, template="model",)
prepared = sft_data.prepare(dataset)

Map:   0%|          | 0/1100 [00:00<?, ? examples/s]

In [16]:
print(prepared[0])
print(sft_data.preview(prepared[0]))

{'prompt': [{'role': 'user', 'content': 'Evaluate the following phrase by transforming it into the spelling given.\n\nInput:\nfreind --> friend'}], 'completion': [{'role': 'assistant', 'content': 'The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".'}]}
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Evaluate the following phrase by transforming it into the spelling given.

Input:
freind --> friend<|im_end|>
<|im_start|>assistant
The spelling of the given phrase "freind" is incorrect, the correct spelling is "friend".<|im_end|>



#### Spliting the data 

In [17]:
train_portion = int(len(data) * 0.85)  # 85% for training
test_portion = int(len(data) * 0.1)    # 10% for testing
val_portion = len(data) - train_portion - test_portion  # Remaining 5% for validation

train_data = prepared.select(range(train_portion))
test_data = prepared.select(
    range(train_portion, train_portion + test_portion)
)
val_data = prepared.select(
    range(train_portion + test_portion, len(prepared))
)

print("Training set length:", len(train_data))
print("Validation set length:", len(val_data))
print("Test set length:", len(test_data))

Training set length: 935
Validation set length: 55
Test set length: 110


### Step-2: - Prepare the Training Set Batches

#### Setting up Tokenizer

In [18]:
# tokenizer = tiktoken.get_encoding("gpt2")
# pad_token_id= tokenizer.encode('<|endoftext|>', allowed_special={"<|endoftext|>"})[0]
# pad_token_id
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

pad_token_id = tokenizer.pad_token_id
pad_token_id

151643

In [19]:
train_dataset = InstructionDataset(
    ds=train_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)


In [20]:
val_dataset = InstructionDataset(
    ds=val_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)
test_dataset = InstructionDataset(
    ds=test_data,
    tokenizer=tokenizer,
    prompt_style=sft_data,
)

#### Sample Data Prepare

In [21]:

# inputs_1 = [0, 1, 2, 3, 4]
# inputs_2 = [5, 6]
# inputs_3 = [7, 8, 9]

# batch = (
#     inputs_1,
#     inputs_2,
#     inputs_3
# )
# custom_collate_fn(batch, pad_token_id=pad_token_id)

In [22]:
# logits_1 = torch.tensor(
#     [[-1.0, 1.0],  # 1st training example
#      [-0.5, 1.5]]  # 2nd training example
# )
# targets_1 = torch.tensor([0, 1])


# loss_1 = torch.nn.functional.cross_entropy(logits_1, targets_1)
# print(loss_1)

# logits_2 = torch.tensor(
#     [[-1.0, 1.0],
#      [-0.5, 1.5],
#      [-0.5, 1.5]]  # New 3rd training example
# )
# targets_2 = torch.tensor([0, 1, 1])

# loss_2 = torch.nn.functional.cross_entropy(logits_2, targets_2)
# print(loss_2)
# targets_3 = torch.tensor([0, 1, -100])

# loss_3 = torch.nn.functional.cross_entropy(logits_2, targets_3)
# print(loss_3)
# print("loss_1 == loss_3:", loss_1 == loss_3)

### Creating, Training and Testing Dataset

In [23]:
from functools import partial
# Changing the defaults at runtime
custom_collate_fn1 = partial(custom_collate_fn, device=device, allowed_max_length = 1024, pad_token_id=pad_token_id)

In [24]:
from torch.utils.data import DataLoader


num_workers = 0
batch_size = 8

torch.manual_seed(123)


train_loader = create_instruction_dataloader(
    train_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=True,
    drop_last=True,
    num_workers=num_workers
)
val_loader = create_instruction_dataloader(
    val_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

test_loader = create_instruction_dataloader(
    test_dataset,
    batch_size=batch_size,
    collate_fn=custom_collate_fn1,
    shuffle=False,
    drop_last=False,
    num_workers=num_workers
)

In [25]:
print("Train loader:")
for inputs, targets in train_loader:
    sample_input , sample_target = inputs, targets
    print(inputs.shape, targets.shape)
    break

Train loader:
torch.Size([8, 59]) torch.Size([8, 59])


### Step-4: - Loading the Model Testing inference

In [26]:
from model_training.framework.model.load_model.gpt2.model import return_model
from model_training.framework.model.load_model.gpt2.configs import load_model_config
from model_training.framework.model.components.inference.inference import gen_sample
from model_training.framework.model.tokenizer.tokenizer_utils import text_to_token_ids, token_ids_to_text

In [27]:
config =load_model_config(dataset_max_len=1024)
# MODEL_NAME = 'gpt2'

model = return_model(MODEL_NAME)

# model = GPT.from_pretrained(config, "gpt2")
# print(f"Model Loaded: {model}")
# model.eval()
# model = model.to(device)

loading wights from pretrained gpt: gpt2-small
{'n_layer': 12, 'n_head': 12, 'emb_dim': 768}


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

In [28]:
# import inspect
# inspect.signature(tokenizer.encode)

In [29]:
model.eval()
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
                              )
in_tokens = in_tokens.to(next(model.parameters()).device)
print(in_tokens.shape)  # [1, sequence_length]

torch.Size([1, 4])


In [30]:
out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

Every effort moves you closer towards a goal. Think of goals in terms of steps: the ultimate goal may be the top of an enormous mountain, but the very first thing you do to start climbing is take your first step


### SFT-Training

In [31]:
import torch.nn.functional as F

model = model.to(device).eval()
sample_input = sample_input.to(device)
sample_target = sample_target.to(device)

with torch.no_grad():
    out = model(input_ids=sample_input)
    loss = F.cross_entropy(
        out.logits.reshape(-1, out.logits.size(-1)),
        sample_target.reshape(-1),
        ignore_index=-100,
    )

print(loss.item())

4.09375


In [32]:
from model_training.framework.train.core.train import train_model, TrainComponents
from model_training.framework.train.core.config import TrainConfig

from model_training.framework.train.loss.sft import sft_forward, sft_loss
from model_training.framework.train.utils.optimizer import configure_optimizer
from model_training.framework.train.utils.lr_scheduler import cosine_lr

In [33]:
seq_length = 1024
total_batch_size = batch_size * seq_length
micro_batch_size = 4
grad_accum_steps = total_batch_size // (micro_batch_size * seq_length )
# seq_length = 1024
# micro_batch_size = train_loader.batch_size  # 1
# grad_accum_steps = 8
num_epochs = 2

In [34]:
train_config = TrainConfig(
    num_steps=(num_epochs * len(train_loader.loader)) // grad_accum_steps,
    learning_rate=1e-5,
    total_batch_size=micro_batch_size * seq_length * grad_accum_steps,
    micro_batch_size=micro_batch_size,
    seq_length=seq_length,
    accumulation_steps=grad_accum_steps,
    max_lr=1e-5,
    min_lr=1e-5,
    warmup_steps=0,
    weight_decay=0.1,
    device=device,
    use_compile=False,
    precision="fp32",
)

In [35]:
torch.manual_seed(123)

optimizer = configure_optimizer(model, train_config.weight_decay, train_config.learning_rate, master_process= True, device=train_config.device)

num decayed parameter tensors: 253, with 3,085,697,024 parameters
num non-decayed parameter tensors: 181, with 241,664 parameters
using fused AdamW: True


In [36]:
from torch.utils.data import DataLoader, Subset
from model_training.framework.data.post_train.dataloader import CyclingDataLoader


debug_examples = 80  # 10 batches × batch_size 8

base_loader = train_loader.loader

small_loader = DataLoader(
    Subset(base_loader.dataset, range(debug_examples)),
    batch_size=base_loader.batch_size,
    collate_fn=base_loader.collate_fn,
    shuffle=True,
    drop_last=True,
    num_workers=base_loader.num_workers,
)

train_loader_subset = CyclingDataLoader(
    small_loader,
    seq_len=train_loader.seq_len,
)

In [37]:
train_comp = TrainComponents(optimizer=optimizer, 
                            dataloader=train_loader, 
                            scheduler= cosine_lr, 
                            forward_step=sft_forward,
                            loss_fn=sft_loss
                            )
train_comp

TrainComponents(optimizer=AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 1e-05
    maximize: False
    weight_decay: 0.1

Parameter Group 1
    amsgrad: False
    betas: (0.9, 0.95)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: True
    lr: 1e-05
    maximize: False
    weight_decay: 0.0
), dataloader=<model_training.framework.data.post_train.dataloader.CyclingDataLoader object at 0x74a73064b230>, scheduler=<function cosine_lr at 0x74a50f0bb560>, forward_step=<function sft_forward at 0x74a50f0bb740>, loss_fn=<function sft_loss at 0x74a50f0bb7e0>)

#### Step-7:- Sample Run 

In [38]:
print("[STAGE] SFT forward and loss")
full_batch = train_comp.dataloader.next_batch()
small_batch = (full_batch[0][:1, :64], full_batch[1][:1, :64])

[STAGE] SFT forward and loss


In [39]:
logits, labels = train_comp.forward_step(model, small_batch, device)

In [40]:
loss = train_comp.loss_fn(logits, labels)

In [41]:
assert logits.shape[:2] == labels.shape
assert loss.ndim == 0 and loss.requires_grad and torch.isfinite(loss)
print(f"✓ logits={tuple(logits.shape)} loss={loss.item():.4f} requires_grad={loss.requires_grad}")

✓ logits=(1, 58, 151936) loss=3.5000 requires_grad=True


### Final Run

In [42]:
torch.set_float32_matmul_precision('high') ## TF16 -> Model Training 

In [43]:
print(train_config.accumulation_steps)

2


In [44]:
from datetime import datetime
from model_training.framework.train.tracking.logger import ExperimentLogger


run_name = datetime.now().strftime("%Y%m%d_%H%M%S")

logger = ExperimentLogger(
    project="sft-experiments",
    log_dir=str(project_root / "execution_results" / "runs" / run_name))

In [45]:
# %tensorboard --logdir "{project_root}/execution_results/runs"

In [46]:
model = train_model(model, train_components=train_comp, train_config=train_config, logger = logger)

Current Device: cuda
Model set to Training mode:True
step 0 | loss: 3.906250 | norm :382.00 | dt: 704.82 ms | tok/sec:23245.54
step 1 | loss: 1.929688 | norm :143.00 | dt: 463.53 ms | tok/sec:35346.05
step 2 | loss: 1.484375 | norm :104.50 | dt: 488.82 ms | tok/sec:33517.54
step 3 | loss: 1.125000 | norm :100.00 | dt: 448.38 ms | tok/sec:36540.67
step 4 | loss: 0.960938 | norm :88.00 | dt: 463.62 ms | tok/sec:35339.47
step 5 | loss: 0.941406 | norm :66.00 | dt: 466.77 ms | tok/sec:35100.50
step 6 | loss: 0.609375 | norm :29.62 | dt: 477.12 ms | tok/sec:34339.20
step 7 | loss: 0.695312 | norm :14.31 | dt: 483.16 ms | tok/sec:33910.17
step 8 | loss: 0.644531 | norm :10.25 | dt: 501.19 ms | tok/sec:32690.29
step 9 | loss: 0.570312 | norm :10.44 | dt: 502.65 ms | tok/sec:32595.17
step 10 | loss: 0.601562 | norm :10.06 | dt: 472.36 ms | tok/sec:34685.49
step 11 | loss: 0.457031 | norm :8.12 | dt: 460.54 ms | tok/sec:35575.99
step 12 | loss: 0.531250 | norm :6.94 | dt: 443.04 ms | tok/sec:36

In [47]:
# text_1 = "What is your name ? "
text_1 = "Every effort moves you"
in_tokens = text_to_token_ids(text_1, tokenizer=tokenizer, 
                            #   allowed_special = {"<|endoftext|>"}
)
in_tokens=in_tokens.to(device)
print(in_tokens.shape)  # [1, sequence_length]


out_tokens = gen_sample(model, in_tokens, max_new_tokens=40, context_len=1024)
print(token_ids_to_text(out_tokens, tokenizer))

torch.Size([1, 4])
Every effort moves you toward your goals, even if you have not reached perfection.  To succeed, you need to have:
A) a goal
Noun phrase (NP) headed by Verb (V).
The clause
